In [6]:
import pandas as pd
import numpy as np
import pandasql as ps

In [7]:
# Import data
validator_metadata = pd.read_csv('../data/validator_metadata.csv', usecols=['validator_index', 'activation_epoch', 'withdrawal_address', 'deposit_address', 'second_deposit_address'])
deposit_address_table = pd.read_csv('../data/validator_metadata.csv', usecols=['validator_index', 'deposit_address', 'second_deposit_address'])
pool_categories = pd.read_csv('../data/pool_categories.csv')
validator_entities = pd.read_csv('../data/validator_entities.csv')
validator_entities_2 = pd.read_csv('../data/validator_metadata.csv', usecols=['validator_index', 'pool']).dropna()

/var/folders/mb/5hm6pgrs3zj_1m_kgvpt40jw0000gn/T/ipykernel_97131/944619681.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  validator_metadata = pd.read_csv('data/validator_metadata.csv', usecols=['validator_index', 'activation_epoch', 'withdrawal_address', 'deposit_address', 'second_deposit_address'])
/var/folders/mb/5hm6pgrs3zj_1m_kgvpt40jw0000gn/T/ipykernel_97131/944619681.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  deposit_address_table = pd.read_csv('data/validator_metadata.csv', usecols=['validator_index', 'deposit_address', 'second_deposit_address'])
/var/folders/mb/5hm6pgrs3zj_1m_kgvpt40jw0000gn/T/ipykernel_97131/944619681.py:6: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  validator_entities_2 = pd.read_csv('data/validator_metadata.csv', usecols=['validator_index', 'pool']).dropna()


In [8]:
# Fill validator pool information from Dune and Rated, using Dune as the first option.
validator_entities_dict = dict(zip(validator_entities['deposit_address'], validator_entities['pool']))
validator_entities_2_dict = dict(zip(validator_entities_2['validator_index'], validator_entities_2['pool']))

for index, row in validator_metadata.iterrows():
    if row['withdrawal_address'] in validator_entities_dict:
        validator_metadata.at[index, 'pool'] = validator_entities_dict[row['withdrawal_address']]
    elif row['deposit_address'] in validator_entities_dict:
        validator_metadata.at[index, 'pool'] = validator_entities_dict[row['deposit_address']]
    elif row['second_deposit_address'] in validator_entities_dict:
        validator_metadata.at[index, 'pool'] = validator_entities_dict[row['second_deposit_address']]
    elif row['validator_index'] in validator_entities_2_dict:
        validator_metadata.at[index, 'pool'] = validator_entities_2_dict[row['validator_index']]
    else: 
        validator_metadata.at[index, 'pool'] = 'Unidentified'
        
validator_metadata

,validator_index,activation_epoch,deposit_address,second_deposit_address,withdrawal_address,pool
0,1391595,NaN,0x30e3056a42d84f9998f3cfc57db63373fc882b26,NaN,0x21ea7e40c9d782749f0eecac020d81f7b6c59cfd,Stader - Permissionless
1,1391700,NaN,0x30e3056a42d84f9998f3cfc57db63373fc882b26,NaN,0xca8a8a972e4cdd757196c3b4194df251f0fbb299,Stader - Permissionless
2,396161,NaN,0x348418ef81e175b5487ecac889a3b96d3d2ab2d8,NaN,0x5945bfe76789c79f54c634f6f704d5400491c90a,pSTAKE
3,396158,NaN,0x348418ef81e175b5487ecac889a3b96d3d2ab2d8,NaN,0x5945bfe76789c79f54c634f6f704d5400491c90a,pSTAKE
4,18846,0.0,0x0038598ecb3b308ebc6c6e2c635bacaa3c5298a3,NaN,0xc771172ae08b5fc37b3ac3d445225928de883876,Poloniex
...,...,...,...,...,...,...
1391865,366084,116722.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius
1391866,366148,116734.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius
1391867,366146,116734.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius
1391868,366159,116737.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius


In [9]:
# Clean up validator metadata
validator_metadata = validator_metadata[validator_metadata['activation_epoch'].notnull()]
validator_metadata.loc[:, 'pool'] = validator_metadata['pool'].fillna('Unidentified')

validator_metadata

,validator_index,activation_epoch,deposit_address,second_deposit_address,withdrawal_address,pool
4,18846,0.0,0x0038598ecb3b308ebc6c6e2c635bacaa3c5298a3,NaN,0xc771172ae08b5fc37b3ac3d445225928de883876,Poloniex
5,18891,0.0,0x0038598ecb3b308ebc6c6e2c635bacaa3c5298a3,NaN,0xc771172ae08b5fc37b3ac3d445225928de883876,Poloniex
6,16368,0.0,0x197939c1ca20c2b506d6811d8b6cdb3394471074,NaN,0x1cd9f4e1449840aec32686e3b20d0f7b104562ed,Cream Finance
7,537,0.0,0x1db3439a222c519ab44bb1144fc28167b4fa6ee6,NaN,NaN,Vitalik Buterin
8,559,0.0,0x1db3439a222c519ab44bb1144fc28167b4fa6ee6,NaN,NaN,Vitalik Buterin
...,...,...,...,...,...,...
1391865,366084,116722.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius
1391866,366148,116734.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius
1391867,366146,116734.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius
1391868,366159,116737.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius


In [10]:
# Fill missing pools based on withdrawal and deposit addresses
# Filter out unidentified pools and missing addresses
withdrawal_pool = validator_metadata[validator_metadata['withdrawal_address'].notna() & (validator_metadata['pool'] != 'Unidentified')]
deposit_pool = validator_metadata[(validator_metadata['pool'] != 'Unidentified')]
deposit_pool_2 = deposit_pool[deposit_pool['second_deposit_address'].notna()]

# Create pool dictionaries
withdrawal_pool_dict = dict(zip(withdrawal_pool['withdrawal_address'], withdrawal_pool['pool']))
deposit_pool_dict = dict(zip(deposit_pool['deposit_address'], deposit_pool['pool']))
deposit_pool_dict_2 = dict(zip(deposit_pool_2['second_deposit_address'], deposit_pool_2['pool']))

# Add pool information from validator_entities
for index, row in validator_entities.iterrows():
    deposit_address = row['deposit_address']
    pool = row['pool']
    if deposit_address not in deposit_pool_dict:
        deposit_pool_dict[deposit_address] = pool
    if deposit_address not in deposit_pool_dict_2:
        deposit_pool_dict_2[deposit_address] = pool

# Update the pool column in the original data frame
for index, row in validator_metadata.iterrows():
    if row['withdrawal_address'] in withdrawal_pool_dict:
        validator_metadata.at[index, 'pool'] = withdrawal_pool_dict[row['withdrawal_address']]
    elif row['deposit_address'] in deposit_pool_dict:
        validator_metadata.at[index, 'pool'] = deposit_pool_dict[row['deposit_address']]
    elif row['second_deposit_address'] in deposit_pool_dict_2:
        validator_metadata.at[index, 'pool'] = deposit_pool_dict_2[row['second_deposit_address']]
    else:
        validator_metadata.at[index, 'pool'] = "Unidentified"

# Merge metadata with pool categories
validator_metadata = pd.merge(validator_metadata, pool_categories, on='pool', how='left')

validator_metadata

,validator_index,activation_epoch,deposit_address,second_deposit_address,withdrawal_address,pool,category
0,18846,0.0,0x0038598ecb3b308ebc6c6e2c635bacaa3c5298a3,NaN,0xc771172ae08b5fc37b3ac3d445225928de883876,Poloniex,CEX
1,18891,0.0,0x0038598ecb3b308ebc6c6e2c635bacaa3c5298a3,NaN,0xc771172ae08b5fc37b3ac3d445225928de883876,Poloniex,CEX
2,16368,0.0,0x197939c1ca20c2b506d6811d8b6cdb3394471074,NaN,0x1cd9f4e1449840aec32686e3b20d0f7b104562ed,Cream Finance,Liquid Staking
3,537,0.0,0x1db3439a222c519ab44bb1144fc28167b4fa6ee6,NaN,NaN,Vitalik Buterin,Solo Stakers
4,559,0.0,0x1db3439a222c519ab44bb1144fc28167b4fa6ee6,NaN,NaN,Vitalik Buterin,Solo Stakers
...,...,...,...,...,...,...,...
1391510,366084,116722.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius,NaN
1391511,366148,116734.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius,NaN
1391512,366146,116734.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius,NaN
1391513,366159,116737.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius,NaN


In [11]:
# Update the pool_size column based on pool and withdrawal_address
validator_metadata['pool_size'] = validator_metadata.groupby('pool')['validator_index'].transform('count')
validator_metadata.loc[validator_metadata['pool'] == 'Unidentified', 'pool_size'] = validator_metadata.groupby('withdrawal_address')['validator_index'].transform('count')
validator_metadata

,validator_index,activation_epoch,deposit_address,second_deposit_address,withdrawal_address,pool,category,pool_size
0,18846,0.0,0x0038598ecb3b308ebc6c6e2c635bacaa3c5298a3,NaN,0xc771172ae08b5fc37b3ac3d445225928de883876,Poloniex,CEX,1371.0
1,18891,0.0,0x0038598ecb3b308ebc6c6e2c635bacaa3c5298a3,NaN,0xc771172ae08b5fc37b3ac3d445225928de883876,Poloniex,CEX,1371.0
2,16368,0.0,0x197939c1ca20c2b506d6811d8b6cdb3394471074,NaN,0x1cd9f4e1449840aec32686e3b20d0f7b104562ed,Cream Finance,Liquid Staking,786.0
3,537,0.0,0x1db3439a222c519ab44bb1144fc28167b4fa6ee6,NaN,NaN,Vitalik Buterin,Solo Stakers,218.0
4,559,0.0,0x1db3439a222c519ab44bb1144fc28167b4fa6ee6,NaN,NaN,Vitalik Buterin,Solo Stakers,218.0
...,...,...,...,...,...,...,...,...
1391510,366084,116722.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius,NaN,25625.0
1391511,366148,116734.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius,NaN,25625.0
1391512,366146,116734.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius,NaN,25625.0
1391513,366159,116737.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius,NaN,25625.0


In [12]:
# Calculate pool sizes based on deposit_address for validators with no identified pool or withdrawal address
# Combine deposit_address and second_deposit_address into a single field
deposit_address_table['combined_addresses'] = deposit_address_table.apply(lambda x: tuple(sorted([x['deposit_address'], x['second_deposit_address']] if pd.notna(x['second_deposit_address']) else [x['deposit_address']])), axis=1)

# Group by the combined_addresses field, count the number of validator indexes, and reset the index
grouped = deposit_address_table.groupby('combined_addresses').size().reset_index(name='pool_size')

# Merge the pool_size back into the original data frame
deposit_address_table = deposit_address_table.merge(grouped, on='combined_addresses', how='left')

# Drop the combined_addresses column
deposit_address_table.drop('combined_addresses', axis=1, inplace=True)

deposit_address_table

,validator_index,deposit_address,second_deposit_address,pool_size
0,1391595,0x30e3056a42d84f9998f3cfc57db63373fc882b26,NaN,299
1,1391700,0x30e3056a42d84f9998f3cfc57db63373fc882b26,NaN,299
2,396161,0x348418ef81e175b5487ecac889a3b96d3d2ab2d8,NaN,9
3,396158,0x348418ef81e175b5487ecac889a3b96d3d2ab2d8,NaN,9
4,18846,0x0038598ecb3b308ebc6c6e2c635bacaa3c5298a3,NaN,400
...,...,...,...,...
1391865,366084,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,11237
1391866,366148,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,11237
1391867,366146,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,11237
1391868,366159,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,11237


In [13]:
validator_metadata = pd.merge(validator_metadata, deposit_address_table[['validator_index', 'pool_size']], on='validator_index', how='left')
validator_metadata['pool_size'] = validator_metadata['pool_size_x'].combine_first(validator_metadata['pool_size_y'])
validator_metadata.drop(columns=['pool_size_x', 'pool_size_y'], inplace=True)

# Define the bins and labels for pool sizes
bins = [0, 2, 6, 20, 100, float('inf')]
labels = ['1', '2-5', '6-19', '20-99', '100+']

# Create the 'pool_size_label' column
validator_metadata['pool_size_label'] = pd.cut(validator_metadata['pool_size'], bins=bins, labels=labels, right=False)
validator_metadata['pool_size_label'] = validator_metadata['pool_size_label'].astype(str)
validator_metadata

,validator_index,activation_epoch,deposit_address,second_deposit_address,withdrawal_address,pool,category,pool_size,pool_size_label
0,18846,0.0,0x0038598ecb3b308ebc6c6e2c635bacaa3c5298a3,NaN,0xc771172ae08b5fc37b3ac3d445225928de883876,Poloniex,CEX,1371.0,100+
1,18891,0.0,0x0038598ecb3b308ebc6c6e2c635bacaa3c5298a3,NaN,0xc771172ae08b5fc37b3ac3d445225928de883876,Poloniex,CEX,1371.0,100+
2,16368,0.0,0x197939c1ca20c2b506d6811d8b6cdb3394471074,NaN,0x1cd9f4e1449840aec32686e3b20d0f7b104562ed,Cream Finance,Liquid Staking,786.0,100+
3,537,0.0,0x1db3439a222c519ab44bb1144fc28167b4fa6ee6,NaN,NaN,Vitalik Buterin,Solo Stakers,218.0,100+
4,559,0.0,0x1db3439a222c519ab44bb1144fc28167b4fa6ee6,NaN,NaN,Vitalik Buterin,Solo Stakers,218.0,100+
...,...,...,...,...,...,...,...,...,...
1391510,366084,116722.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius,NaN,25625.0,100+
1391511,366148,116734.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius,NaN,25625.0,100+
1391512,366146,116734.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius,NaN,25625.0,100+
1391513,366159,116737.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,Celsius,NaN,25625.0,100+


In [14]:
# Add missing columns back to the data frame and reorder the columns
missing_columns = pd.read_csv('../data/validator_metadata.csv', usecols=['validator_index', 'validator_pubkey', 'exit_epoch'])
validator_metadata = pd.merge(validator_metadata, missing_columns, on='validator_index', how='left')
validator_metadata = validator_metadata[['validator_index', 'pool', 'category', 'pool_size', 'pool_size_label', 'activation_epoch', 'exit_epoch', 'withdrawal_address', 'deposit_address', 'second_deposit_address']]

validator_metadata

,validator_index,pool,category,pool_size,pool_size_label,activation_epoch,exit_epoch,withdrawal_address,deposit_address,second_deposit_address
0,18846,Poloniex,CEX,1371.0,100+,0.0,252781.0,0xc771172ae08b5fc37b3ac3d445225928de883876,0x0038598ecb3b308ebc6c6e2c635bacaa3c5298a3,NaN
1,18891,Poloniex,CEX,1371.0,100+,0.0,252783.0,0xc771172ae08b5fc37b3ac3d445225928de883876,0x0038598ecb3b308ebc6c6e2c635bacaa3c5298a3,NaN
2,16368,Cream Finance,Liquid Staking,786.0,100+,0.0,194722.0,0x1cd9f4e1449840aec32686e3b20d0f7b104562ed,0x197939c1ca20c2b506d6811d8b6cdb3394471074,NaN
3,537,Vitalik Buterin,Solo Stakers,218.0,100+,0.0,NaN,NaN,0x1db3439a222c519ab44bb1144fc28167b4fa6ee6,NaN
4,559,Vitalik Buterin,Solo Stakers,218.0,100+,0.0,NaN,NaN,0x1db3439a222c519ab44bb1144fc28167b4fa6ee6,NaN
...,...,...,...,...,...,...,...,...,...,...
1391510,366084,Celsius,NaN,25625.0,100+,116722.0,242454.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN
1391511,366148,Celsius,NaN,25625.0,100+,116734.0,248692.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN
1391512,366146,Celsius,NaN,25625.0,100+,116734.0,248723.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN
1391513,366159,Celsius,NaN,25625.0,100+,116737.0,248754.0,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,0xeee27662c2b8eba3cd936a23f039f3189633e4c8,NaN


In [15]:
# Check for missing values
print(validator_metadata['validator_index'].isna().any())
print(validator_metadata['pool'].isna().any())
print(validator_metadata['category'].isna().any())
print(validator_metadata['pool_size'].isna().any())
print(validator_metadata['pool_size_label'].isna().any())
print(validator_metadata.dtypes)

False
False
True
False
False
validator_index             int64
pool                       object
category                   object
pool_size                 float64
pool_size_label            object
activation_epoch          float64
exit_epoch                float64
withdrawal_address         object
deposit_address            object
second_deposit_address     object
dtype: object


In [ ]:
# Export the table
validator_metadata.to_csv('../int/validator_metadata_v2.csv', index=False)